# 05 · Deep dives
Root-cause analysis, cohorts, RFM, basket and price, marketing funnel and inventory. Each section states the method, shows the result with its
uncertainty and ends with a so-what. Numbers that overlap `docs/reconciliation.md` are asserted against it.

**Contents:** A. Root cause: lateness, poor experience, revenue variance - B. Cohorts - C. RFM and Pareto - D. Basket and price - E. Marketing funnel - F. Inventory - G. Analyses considered and rejected.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
from scipy import stats
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from IPython.display import display
from viz_style import *
from data_io import *
o = orders_enriched(); mk = clean("fact_marketing"); inv = clean("fact_inventory"); pr = clean("dim_product")
H = headline_numbers(o, mk, inv)
END = o.order_date.max()
rng = np.random.default_rng(2026)
N = len(o); late_rate = o.is_late.mean()
assert N == 5000 and round(late_rate, 3) == .306 and round(o.poor_experience.mean(), 3) == .458

## A1. Root cause of lateness: what separates late orders from on-time ones?
**Method.** Late rate (status not On Time, 30.6%) by distance band, time of day, weekday, category and payment with Wilson 95% CIs; chi-square test per factor
(area has 316 levels, so it is shown as a funnel plot against binomial limits); a logistic model on all drivers with a likelihood-ratio test, and a cross-validated AUC
as a predictability ceiling (0.5 = no better than a coin).

In [2]:
FACT = {"Distance band": "dist_band", "Time of day": "day_part", "Weekday": "dow_name", "Category": "category", "Payment": "payment_method"}
ORDER = {"dow_name": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]}
tests, tabs = {}, {}
for name, col in FACT.items():
    t = pd.crosstab(o[col], o.is_late)
    if col in ORDER:
        t = t.reindex(ORDER[col])
    chi2, p, dof, _ = stats.chi2_contingency(t)
    tests[name] = (p, chi2, dof, np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1))))
    k, n = t[True].values, t.sum(1).values
    lo, hi = wilson(k, n)
    tabs[name] = pd.DataFrame({"late": k, "n": n, "rate": k / n, "lo": lo, "hi": hi}, index=t.index.astype(str))
area = o.groupby("area").is_late.agg(["sum", "size"])
p_area = stats.chi2_contingency(pd.crosstab(o.area, o.is_late))[1]
d = o.assign(late=o.is_late.astype(int))
full = smf.logit("late ~ distance_km + C(day_part) + C(dow_name) + C(category) + C(payment_method)", d).fit(disp=0)
X = pd.get_dummies(o[["day_part", "dow_name", "category", "payment_method", "area"]].astype(str), drop_first=True).assign(distance=o.distance_km.values)
cv = StratifiedKFold(5, shuffle=True, random_state=1)
auc_late = roc_auc_score(o.is_late, cross_val_predict(LogisticRegression(max_iter=2000, C=0.05), X, o.is_late, cv=cv, method="predict_proba")[:, 1])
y_poor = o.poor_experience
auc_poor = roc_auc_score(y_poor, cross_val_predict(LogisticRegression(max_iter=2000, C=0.05), X, y_poor, cv=cv, method="predict_proba")[:, 1])
print({k: f"p={v[0]:.2f}, V={v[3]:.3f}" for k, v in tests.items()}, f"| area p={p_area:.2f}")
print(f"logit LR test p = {full.llr_pvalue:.2f} | pseudo R2 = {full.prsquared:.4f} | CV AUC late = {auc_late:.3f}, poor experience = {auc_poor:.3f}")
assert min(v[0] for v in tests.values()) > 0.05 and full.llr_pvalue > .05 and auc_late < .56

{'Distance band': 'p=0.89, V=0.011', 'Time of day': 'p=0.82, V=0.017', 'Weekday': 'p=0.89, V=0.022', 'Category': 'p=0.20, V=0.052', 'Payment': 'p=0.56, V=0.020'} | area p=0.45
logit LR test p = 0.72 | pseudo R2 = 0.0032 | CV AUC late = 0.485, poor experience = 0.502


In [3]:
fig, axs = new_fig(h=6.6, nrows=2, ncols=3, left=0.15)
fig.subplots_adjust(wspace=.8, hspace=.28)
for ax, (name, tab) in zip(axs.flat, tabs.items()):
    yy = np.arange(len(tab))
    ax.errorbar(tab.rate * 100, yy, xerr=[(tab.rate - tab.lo) * 100, (tab.hi - tab.rate) * 100], fmt="o", color=GREEN, capsize=2.5, ms=5, lw=1.2)
    ax.axvline(late_rate * 100, color=GREY, ls="--", lw=1)
    ax.set_yticks(yy, [f"{i} ({n})" for i, n in zip(tab.index, tab.n)], fontsize=7.5); ax.invert_yaxis(); ax.set_xlim(15, 50); ax.set_xlabel("late rate, %", fontsize=8)
    clean_axes(ax, "x"); ax.set_title(f"{name}: {pfmt(tests[name][0])}", fontsize=9.5)
ax = axs.flat[5]
ar = area[area["size"] >= 3]
ax.scatter(ar["size"], ar["sum"] / ar["size"] * 100, s=9, alpha=.45, color=GREEN, linewidths=0)
nn = np.arange(3, ar["size"].max() + 1)
ax.plot(nn, (late_rate + 1.96 * np.sqrt(late_rate * (1 - late_rate) / nn)) * 100, color=GREY, lw=1, ls="--"); ax.plot(nn, (late_rate - 1.96 * np.sqrt(late_rate * (1 - late_rate) / nn)) * 100, color=GREY, lw=1, ls="--")
outside = ((ar["sum"] / ar["size"] > late_rate + 1.96 * np.sqrt(late_rate * (1 - late_rate) / ar["size"])) | (ar["sum"] / ar["size"] < late_rate - 1.96 * np.sqrt(late_rate * (1 - late_rate) / ar["size"]))).mean()
ax.set_xlabel("Orders in the area"); ax.set_ylim(-3, 103); clean_axes(ax, "y"); ax.set_title(f"Area ({len(area)}): {pfmt(p_area)}", fontsize=9.5); ax.set_ylabel("late rate, % (funnel plot)", fontsize=8)
note(ax, .97, .95, f"{outside:.0%} outside the 95% limits\n(5% expected by chance)", transform=ax.transAxes, ha="right", va="top", fontsize=8)
headline(fig, f"No driver explains lateness: it is about {late_rate:.0%} in every distance band, time of day, weekday, category, payment method and area",
         f"Late rate (95% Wilson CI), dashed line = overall {late_rate:.1%}. Logistic model on all drivers: likelihood-ratio {pfmt(full.llr_pvalue)}; cross-validated AUC {auc_late:.2f} (0.5 = coin flip)")
footnote(fig, "Source: data/clean fact_orders, dim_customer, dim_product. Late = delivery status not On Time. Distance, hour and area are unrelated to lateness, which points to an ETA that ignores them (to verify on real data).")
print(save(fig, "ds/rca_lateness_drivers"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\rca_lateness_drivers.png


**Contribution view.** If one factor were the cause, some of its levels would carry many more late orders than the overall rate predicts. The waterfall shows that for the
factor with the smallest p-value: each level's late orders minus the number expected at 30.6%; the levels net to zero by construction, whiskers show the 95% noise band.

In [4]:
best = min(tests, key=lambda k: tests[k][0])
tb = tabs[best].copy()
tb["excess"] = tb.late - tb.n * late_rate
tb["noise"] = 1.96 * np.sqrt(tb.n * late_rate * (1 - late_rate))
tb = tb.sort_values("excess", ascending=False)
print(best, "| largest gap", tb.excess.idxmax(), round(tb.excess.max(), 1), "orders, noise band +/-", round(tb.noise[tb.excess.idxmax()], 1), "| late orders", int(o.is_late.sum()))
fig, ax = new_fig(h=4.6, left=0.05)
import textwrap
waterfall(ax, [textwrap.fill(i, 11) + chr(10) + f"(n = {n})" for i, n in zip(tb.index, tb.n)], tb.excess.values, end_label="Net", fmt=lambda v: f"{v:+.0f}", total_fmt=lambda v: f"{v:+.0f}",
          err=[(v - h, v + h) for v, h in zip(tb.excess, tb.noise)])
ax.axhline(0, color=INK, lw=.8)
plt.setp(ax.get_xticklabels(), fontsize=7.5)
mx = tb.excess.idxmax()
n_out = int((tb.excess.abs() > tb.noise).sum())
headline(fig, f"By {best.lower()}, {n_out} of {len(tb)} levels sit outside their noise band (about {len(tb) * .05:.1f} expected by chance); the largest gap is {tb.excess.max():+.0f} late orders, {tb.excess.max() / o.is_late.sum():.1%} of all late orders",
         f"Late orders above (+) or below (-) the {late_rate:.1%} expectation, per {best.lower()} level ({best}: omnibus chi-square {pfmt(tests[best][0])}, the smallest of the five factors); whiskers = 95% noise band per level, not corrected for {len(tb)} comparisons")
footnote(fig, f"Source: data/clean fact_orders. Total late orders {int(o.is_late.sum()):,}. Levels net to zero by construction. A single level near its band is what comparing {len(tb)} levels produces; the omnibus test does not reject equal rates.")
print(save(fig, "ds/rca_lateness_contribution"))

Category | largest gap Snacks & Munchies 23.2 orders, noise band +/- 19.9 | late orders 1530
D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\rca_lateness_contribution.png


**So what.** Lateness looks like a flat 30% tax on every slice, so a fix aimed at one slice (a zone, an hour, a category) has nothing to hit. In real data the first checks
would be rider supply by hour and the ETA promise by distance; here both are flat, a sign that promised times are generated independently of the delivery.

## A2. Root cause of a poor experience: late only, low rating only, or both?
**Method.** Split the 45.8% poor-experience orders into three exclusive groups, test whether lateness and low rating are independent (chi-square), then compare each group's share
across factor levels (binomial test per cell; about 5% of 72 cells would flag by chance).

In [5]:
gx = o.experience.value_counts().reindex(["Good", "Late only", "Low rating only", "Both"])
rev = o.groupby("experience").order_total.sum().reindex(gx.index)
exp_both = o.is_late.mean() * o.low_rating.mean() * N
ct = pd.crosstab(o.is_late, o.low_rating); chi_ind, p_ind = stats.chi2_contingency(ct)[:2]
print(gx.to_dict(), "| both observed", int(gx["Both"]), "expected if independent", round(exp_both), "| independence p =", round(p_ind, 2))
assert round(rev[["Late only", "Low rating only", "Both"]].sum(), 2) == round(H["rar"], 2) == 5042193.50
FAC2 = {"Payment": "payment_method", "Distance": "dist_band", "Time of day": "day_part", "Category": "category"}
rows, flags, pmin = {}, {}, {}
for name, col in FAC2.items():
    t = pd.crosstab(o[col], o.experience); pmin[name] = stats.chi2_contingency(t)[1]
    for lev in t.index:
        r_ = {}
        for g in ["Late only", "Low rating only", "Both"]:
            base = (o.experience == g).mean(); k, n = t.loc[lev, g], t.loc[lev].sum()
            r_[g] = (k / n - base) * 100
            flags[(name, str(lev), g)] = stats.binomtest(int(k), int(n), base).pvalue < .05
        rows[(name, str(lev))] = r_
Dv = pd.DataFrame(rows).T
Fl = pd.Series(flags).unstack().reindex(Dv.index.map(lambda x: x), fill_value=False) if False else None
flag_df = pd.DataFrame({g: [flags[(a, b, g)] for a, b in Dv.index] for g in Dv.columns}, index=Dv.index)
print("flagged cells", int(flag_df.values.sum()), "of", flag_df.size, "| min chi-square p by factor", {k: round(v, 2) for k, v in pmin.items()})

{'Good': 2711, 'Late only': 1211, 'Low rating only': 759, 'Both': 319} | both observed 319 expected if independent 330 | independence p = 0.44
flagged cells 3 of 72 | min chi-square p by factor {'Payment': np.float64(0.94), 'Distance': np.float64(0.18), 'Time of day': np.float64(0.82), 'Category': np.float64(0.34)}


In [6]:
fig, (a1, a2) = new_fig(h=6.4, ncols=2, gridspec_kw={"width_ratios": [1, 1.25]}, left=0.15)
share = gx / N
cols_ = [LTGREEN, YELLOW, "#E8B820", "#B98300"]
a1.barh(range(4), share.values, color=[GREEN, YELLOW, MIDGREEN, "#B98300"], height=.62)
for i, (g, v) in enumerate(share.items()):
    lo, hi = wilson(gx[g], N)
    a1.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt="none", ecolor=INK, capsize=3, elinewidth=1)
    a1.text(hi + .01, i, f"{v:.1%}" + chr(10) + f"{gx[g]:,} orders, {inr(rev[g])}", va="center", fontsize=8.5)
a1.set_yticks(range(4), ["Good", "Late only", "Low rating only", "Both late and 1-2 stars"]); a1.invert_yaxis(); a1.set_xlim(0, .95); a1.xaxis.set_visible(False)
a1.spines[["bottom", "left"]].set_visible(False); a1.tick_params(length=0)
a1.set_title(f"Share of orders. Both = {gx['Both']} vs {exp_both:.0f} expected if independent ({pfmt(p_ind)})", fontsize=9)
lab = [f"{a}: {b}" for a, b in Dv.index]
ann = Dv.map(lambda v: f"{v:+.1f}") + flag_df.map(lambda f: "*" if f else "")
sns.heatmap(Dv, ax=a2, cmap=DIV, center=0, vmin=-8, vmax=8, annot=ann, fmt="", annot_kws={"fontsize": 7}, linewidths=.5, linecolor="white", cbar_kws={"shrink": .6, "label": "points vs overall share"})
a2.set_yticklabels([l.replace("_", " ") for l in lab], rotation=0, fontsize=7.5); a2.set_ylabel(""); a2.set_xlabel(""); a2.tick_params(length=0); a2.xaxis.tick_top()
a2.set_title("Over / under-index vs overall share, points (* p < 0.05, one cell)", fontsize=9, pad=30)
headline(fig, f"Poor experience is two independent problems, not one: {share['Late only']:.0%} late only, {share['Low rating only']:.0%} low rating only, {share['Both']:.0%} both",
         f"45.8% of orders ({inr(H['rar'])}, the Revenue at Risk) are poor. Factor chi-square p {min(pmin.values()):.2f} to {max(pmin.values()):.2f}; {int(flag_df.values.sum())} of {flag_df.size} cells flag (about {flag_df.size * .05:.0f} by chance)")
footnote(fig, "Source: data/clean fact_orders. Low rating = 1-2 stars. 'Both' matches independence (lateness does not predict low ratings), so Revenue at Risk adds two separate exposures and fixing delivery alone leaves low ratings intact.")
print(save(fig, "ds/rca_poor_experience"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\rca_poor_experience.png


**So what.** The two causes do not overlap more than chance, and neither concentrates in a slice. Late-only orders are the group a delivery fix can reach (about 1,000 orders); the
low-rating-only group needs a different cause (product, service) that the feedback tags do not identify.

## A3. Revenue variance: volume, mix and AOV
**Method.** Revenue = orders x AOV, with AOV split into category mix and within-category value:
volume = (N1 - N0) x AOV0, mix = N1 x sum((share1 - share0) x AOV0 by category), AOV = N1 x sum(share1 x (AOV1 - AOV0) by category). The three effects add exactly to the revenue change.
Uncertainty: mix and AOV effects by bootstrap (resampling orders within each period), volume by Poisson noise on the order counts.

In [7]:
def effects(a, b):
    N0, N1 = len(a), len(b); aov0 = a.order_total.mean()
    ca = a.groupby("category").order_total.agg(["size", "mean"]); cb = b.groupby("category").order_total.agg(["size", "mean"])
    cats = ca.index.union(cb.index)
    s0 = ca["size"].reindex(cats, fill_value=0) / N0; s1 = cb["size"].reindex(cats, fill_value=0) / N1
    a0 = ca["mean"].reindex(cats).fillna(aov0); a1 = cb["mean"].reindex(cats).fillna(0)
    vol = (N1 - N0) * aov0; mix = N1 * ((s1 - s0) * a0).sum(); rate = N1 * (s1 * (a1 - a0)).sum()
    return np.array([vol, mix, rate])
def bridge(a, b, nb=800):
    e = effects(a, b); tot = b.order_total.sum() - a.order_total.sum()
    assert abs(e.sum() - tot) < 1e-6, (e.sum(), tot)
    bs = []
    for _ in range(nb):
        bs.append(effects(a.sample(len(a), replace=True, random_state=int(rng.integers(1e9))), b.sample(len(b), replace=True, random_state=int(rng.integers(1e9)))))
    bs = np.array(bs); ci = np.percentile(bs, [2.5, 97.5], axis=0).T
    N0, N1 = len(a), len(b); half = 1.96 * np.sqrt(N0 + N1) * a.order_total.mean()
    ci[0] = [e[0] - half, e[0] + half]
    return e, ci, tot
mm = o[o.is_full_month]
def per(lo, hi=None):
    hi = hi or lo
    return mm[(mm.month_start >= lo) & (mm.month_start <= hi)]
MOM = [("Jul to Aug 2024", per("2024-07-01"), per("2024-08-01")), ("Aug to Sep 2024", per("2024-08-01"), per("2024-09-01")), ("Sep to Oct 2024", per("2024-09-01"), per("2024-10-01"))]
base23 = mm[(mm.month_start >= "2023-04-01") & (mm.month_start <= "2023-10-01")]; cur24 = mm[(mm.month_start >= "2024-04-01") & (mm.month_start <= "2024-10-01")]
res_mom = [(t, *bridge(a, b)) for t, a, b in MOM]
e_y, ci_y, tot_y = bridge(base23, cur24)
yoy = cur24.order_total.sum() / base23.order_total.sum() - 1
print("YoY Apr-Oct:", round(yoy, 4), "| change", round(tot_y), "| volume/mix/AOV", np.round(e_y).astype(int).tolist())
assert round(yoy, 4) == -0.0451
mo = (cur24.groupby("month_start").order_total.sum().values / base23.groupby("month_start").order_total.sum().values - 1)
assert [round(v * 100, 1) for v in mo] == [-2.8, -5.6, 6.7, 1.0, -12.4, -9.2, -7.0]
for t, e, ci_, tot in res_mom:
    print(t, round(tot), np.round(e).astype(int).tolist())

YoY Apr-Oct: -0.0451 | change -180688 | volume/mix/AOV [-85047, -2132, -93509]
Jul to Aug 2024 -26917 [-11194, -2441, -13283]
Aug to Sep 2024 -27500 [-8704, 2300, -21096]
Sep to Oct 2024 19008 [0, 4573, 14435]


In [8]:
fig = blank_fig(h=7.4, left=0.07, pad=0.1)
gs = grid(fig, 2, 6, hspace=.5, wspace=1.2)
LB = ["Volume\n(orders)", "Mix\n(category)", "AOV\n(basket)"]
for i, (t, e, ci_, tot) in enumerate(res_mom):
    ax = fig.add_subplot(gs[0, 2 * i:2 * i + 2])
    waterfall(ax, LB, e / 1e5, end_label="Total\nchange", fmt=lambda v: f"{v:+.2f}", total_fmt=lambda v: f"{v:+.2f}", err=[(a_ / 1e5, b_ / 1e5) for a_, b_ in ci_])
    ax.axhline(0, color=INK, lw=.8); ax.set_title(f"{t}: {tot / 1e5:+.2f} L ({tot / per(MOM[i][1].month_start.iloc[0]).order_total.sum():+.1%})", fontsize=9.5)
ax = fig.add_subplot(gs[1, 0:3])
waterfall(ax, LB, e_y / 1e5, end_label="Total\nchange", fmt=lambda v: f"{v:+.2f}", total_fmt=lambda v: f"{v:+.2f}", err=[(a_ / 1e5, b_ / 1e5) for a_, b_ in ci_y])
ax.axhline(0, color=INK, lw=.8); ax.set_title(f"Apr to Oct 2024 vs 2023: {tot_y / 1e5:+.2f} L ({yoy:+.1%})", fontsize=9.5)
ax = fig.add_subplot(gs[1, 3:6])
mlab = [d.strftime("%b") for d in sorted(cur24.month_start.unique())]
ax.bar(range(7), mo * 100, color=[GOOD if v > 0 else BAD for v in mo], width=.6)
for i, v in enumerate(mo * 100): ax.text(i, v + (.6 if v > 0 else -.6), f"{v:+.1f}%", ha="center", va="bottom" if v > 0 else "top", fontsize=8.5)
ax.axhline(0, color=INK, lw=.8); ax.set_xticks(range(7), mlab); ax.set_ylim(-17, 11); ax.yaxis.set_visible(False); ax.spines[["left", "top", "right", "bottom"]].set_visible(False); ax.tick_params(length=0)
ax.set_title("Year on year by month: no consistent direction", fontsize=9.5)
headline(fig, f"Revenue changes are mostly noise: Apr to Oct 2024 is {yoy:+.1%} on 2023, split between fewer orders ({e_y[0] / 1e5:+.2f} L) and a lower AOV ({e_y[2] / 1e5:+.2f} L); neither interval excludes zero",
         "Revenue change in ₹ lakh split into volume (orders), mix (category) and AOV effects, with 95% intervals. Top: last three full month pairs. Bottom: year on year")
footnote(fig, "Source: data/clean fact_orders, full months only (Mar 2023 and Nov 2024 excluded). Bars start at zero change. Category is the mix dimension (order_total is unrelated to the item, so mix is expected to be about 0). YoY matches docs/reconciliation.md (-4.5%).")
print(save(fig, "ds/rca_revenue_bridge"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\rca_revenue_bridge.png


**So what.** A 5% month-on-month move is what 250 orders a month produce by chance: the volume bar is the only one whose interval comes close to excluding zero, and only in the
year-on-year view. Do not explain individual months; track a rolling 3-month total, and read the YoY decline (-4.5%) as weak evidence of fewer orders, not a mix or basket problem.

## B. Cohort analysis
**Method.** Customers are grouped by the month of their first order (the registration date is unusable: 51% of orders precede it). Retention = share of a cohort with an order
in month k after its first order; cohorts are masked where month k lies beyond the data (Nov 2024 is a partial month, as in the data science review). Revenue per acquired customer is cumulated by k.
Time to second order uses Kaplan-Meier because customers without a second order are censored at the data end, not "never".

In [9]:
fo_ = o.groupby("customer_id").order_date.min().rename("first")
c = o.merge(fo_, on="customer_id")
c["cohort"] = c["first"].dt.to_period("M"); c["cm"] = c.order_date.dt.to_period("M")
c["k"] = (c.cm.dt.year - c.cohort.dt.year) * 12 + (c.cm.dt.month - c.cohort.dt.month)
coh = c.groupby(["cohort", "k"]).customer_id.nunique().unstack(fill_value=0)
size = coh[0]
ret = coh.div(size, axis=0).astype(float)
last_m = c.cm.max()
for ch_ in ret.index:
    for k in ret.columns:
        if (ch_ + int(k)) > last_m:
            ret.loc[ch_, k] = np.nan
mean_k = {k: float(ret[k].mean()) for k in range(1, 7)}
print("mean retention M+1..M+6:", {k: round(v, 3) for k, v in mean_k.items()}, "| cohorts", len(ret), "| sizes", int(size.min()), "to", int(size.max()))
for k, ref in zip(range(1, 7), [.085, .094, .082, .098, .09, .083]):
    assert abs(mean_k[k] - ref) < .0015, (k, mean_k[k], ref)
pooled = {k: coh[k][[ch_ for ch_ in coh.index if (ch_ + k) <= last_m]].sum() / size[[ch_ for ch_ in coh.index if (ch_ + k) <= last_m]].sum() for k in range(1, 7)}
print("customer-weighted retention M+1..M+6:", {k: round(float(v), 3) for k, v in pooled.items()})

mean retention M+1..M+6: {1: 0.085, 2: 0.094, 3: 0.082, 4: 0.098, 5: 0.09, 6: 0.083} | cohorts 21 | sizes 5 to 233
customer-weighted retention M+1..M+6: {1: 0.091, 2: 0.095, 3: 0.086, 4: 0.099, 5: 0.097, 6: 0.09}


In [10]:
KMAX = 12
rp = ret.loc[:, 1:KMAX].copy()
rp.index = [f"{p.strftime('%b-%y')}  (n = {int(size[p])})" for p in rp.index]
allrow = ret.loc[:, 1:KMAX].mean().to_frame().T; allrow.index = ["All cohorts, mean"]
M = pd.concat([rp, allrow]) * 100
fig, ax = new_fig(h=6.6, left=0.14)
ann = M.map(lambda v: "" if pd.isna(v) else f"{v:.0f}")
sns.heatmap(M, ax=ax, cmap=SEQ_LIGHT, vmin=0, vmax=30, annot=ann, fmt="", annot_kws={"fontsize": 7.5}, linewidths=.5, linecolor="white", cbar_kws={"shrink": .7, "label": "% of cohort ordering in month k"})
ax.axhline(len(rp), color=INK, lw=1.2); ax.set_xlabel("Months since first order"); ax.set_ylabel(""); ax.tick_params(length=0); plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
ax.set_xticklabels([f"M+{k}" for k in range(1, KMAX + 1)], rotation=0)
headline(fig, f"Retention is flat at about {np.mean(list(mean_k.values())):.0%} in every month after the first order: no decay, no maturing cohort",
         "Share of each first-order-month cohort that orders again k months later (blank = month not yet observed). Cohort size in brackets; scale starts at 0")
footnote(fig, "Source: data/clean fact_orders; cohort = month of first order in the data (Mar 2023 holds customers who may have ordered earlier). Cohorts of 5 to 233 customers: single cells are noisy. Real retention decays; this flat line fits random order assignment.")
print(save(fig, "ds/cohort_retention"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\cohort_retention.png


In [11]:
rev_ck = c.groupby(["cohort", "k"]).order_total.sum().unstack(fill_value=0)
rpc = rev_ck.div(size, axis=0)
for ch_ in rpc.index:
    for k in rpc.columns:
        if (ch_ + int(k)) > last_m:
            rpc.loc[ch_, k] = np.nan
cum = rpc.fillna(0).cumsum(axis=1).where(rpc.notna())
inc = {k: rev_ck[k][[x for x in rev_ck.index if (x + k) <= last_m]].sum() / size[[x for x in rev_ck.index if (x + k) <= last_m]].sum() for k in range(0, 13)}
cum_pool = np.cumsum(list(inc.values()))
print("first order per customer", round(inc[0]), "| later months, mean per customer per month", round(np.mean([inc[k] for k in range(1, 13)])), "| cumulative at M+12", round(cum_pool[12]))

first order per customer 2320 | later months, mean per customer per month 212 | cumulative at M+12 4869


### Time to second order

In [12]:
from statsmodels.duration.survfunc import SurvfuncRight
s2 = o.sort_values(["customer_id", "order_date", "order_id"])
g1 = s2.groupby("customer_id").order_date.agg(["min", lambda x: x.iloc[1] if len(x) > 1 else pd.NaT]); g1.columns = ["first", "second"]
g1["event"] = g1.second.notna().astype(int)
g1["days"] = np.where(g1.event == 1, (g1.second - g1["first"]).dt.days, (END - g1["first"]).dt.days)
obs = g1[g1.event == 1].days
sf = SurvfuncRight(g1.days.values.astype(float), g1.event.values)
t_ = sf.surv_times; cdf = 1 - sf.surv_prob; se = sf.surv_prob_se
med_obs, p90 = obs.median(), obs.quantile(.9)
t_med = float(t_[np.argmax(cdf >= .5)]) if (cdf >= .5).any() else np.nan
print(f"observed second orders n = {len(obs):,}, median {med_obs:.0f} days, P90 {p90:.0f} | KM share reordered by 365 days: {float(np.interp(365, t_, cdf)):.1%} | KM median {t_med}")
assert len(obs) == 1492 and med_obs == 134

observed second orders n = 1,492, median 134 days, P90 362 | KM share reordered by 365 days: 70.1% | KM median 213.0


In [13]:
fig, (a1, a2) = new_fig(h=4.8, ncols=2)
curves = [x for x in cum.index if size[x] >= 40]
for x in curves:
    a1.plot(range(0, 13), cum.loc[x, 0:12].values, color=LTGREY, lw=1)
a1.plot(range(0, 13), cum_pool, color=GREEN, lw=3, label="all cohorts, weighted")
a1.set_xlabel("Months since first order"); a1.set_ylabel("Cumulative revenue per acquired customer, ₹"); a1.set_ylim(0, 5800); clean_axes(a1)
a1.text(12, cum_pool[12], f"  ₹{cum_pool[12]:,.0f}", va="center", fontsize=9, color=GREEN, fontweight="bold")
a1.text(12, 900, "first order ₹%s" % format(inc[0], ",.0f") + chr(10) + "then about ₹%s a month" % format(np.mean([inc[k] for k in range(1, 13)]), ",.0f"), fontsize=9.5, va="bottom", ha="right")
a1.set_title(f"Revenue per customer: {len(curves)} cohorts (grey) and the average")
a2.hist(obs, bins=np.arange(0, 600, 30), color=GREEN, alpha=.85)
a2.axvline(med_obs, color=INK, ls="--", lw=1); a2.text(med_obs + 6, a2.get_ylim()[1] * .95, "median %d days, P90 %d" % (med_obs, p90) + chr(10) + "(customers who reordered)" + chr(10) + "Kaplan-Meier median %d days;" % t_med + chr(10) + "%.0f%% reorder within a year" % (100 * float(np.interp(365, t_, cdf))) + chr(10) + "(counts customers still waiting)", va="top", fontsize=9)
a2.set_xlabel("Days from first to second order (customers who reordered, n = 1,492)"); a2.set_ylabel("Customers"); clean_axes(a2)
a2.set_title(f"Time to second order; {1 - g1.event.mean():.0%} of customers have not reordered yet")
headline(fig, f"A customer's value grows by a steady ₹{np.mean([inc[k] for k in range(1, 13)]):,.0f} a month after the first order, and half of new customers have reordered after about {t_med / 30.4:.0f} months (Kaplan-Meier median {t_med:.0f} days)",
         "Left: cumulative revenue per acquired customer by cohort. Right: distribution of days to the second order (observed reorders only)")
footnote(fig, "Source: data/clean fact_orders. Cohort curves end where the data end, so later months rest on fewer cohorts. The histogram covers observed reorders only (it understates the wait); the Kaplan-Meier median counts customers still waiting.")
print(save(fig, "ds/cohort_revenue_and_second_order"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\cohort_revenue_and_second_order.png


**So what.** There is no cohort effect to manage: every cohort adds about the same revenue per month, and retention does not decay. A reorder cycle of about 4 to 5 months makes a
"60-day reorder" a harsh KPI for groceries; in real data a day-90 win-back nudge is the natural test (see the review).

## C. RFM segmentation and the Pareto curve
**Method.** Recency (days since last order), frequency (orders) and monetary value (revenue) scored in quintiles (5 = best; frequency ties broken by revenue, recency by rank) for the
2,172 ordering customers, then grouped into six actionable segments. To judge whether the segments are real, the same procedure is run 100 times after shuffling which customer each order belongs to.

In [14]:
def rfm(df, end=END):
    r = df.groupby("customer_id").agg(last=("order_date", "max"), f=("order_id", "size"), m=("order_total", "sum"))
    r["r_days"] = (end - r["last"]).dt.days
    r["R"] = pd.qcut(r.r_days.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
    r["F"] = pd.qcut((r.f * 1e7 + r.m).rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    r["M"] = pd.qcut(r.m.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    R_, F_ = r.R, r.F
    r["segment"] = np.select([(R_ >= 4) & (F_ >= 4), (R_ >= 3) & (F_ >= 3), (R_ <= 2) & (F_ >= 3), (R_ >= 4) & (F_ <= 2), (R_ == 3) & (F_ <= 2)],
                             ["Champions", "Loyal", "At risk", "New / recent", "Needs attention"], "Hibernating")
    return r
rf = rfm(o)
SEGS = rf.groupby("segment").agg(customers=("m", "size"), revenue=("m", "sum"), orders=("f", "mean"), recency=("r_days", "mean"))
SEGS["cust_share"] = SEGS.customers / SEGS.customers.sum(); SEGS["rev_share"] = SEGS.revenue / SEGS.revenue.sum()
SEGS = SEGS.sort_values("rev_share", ascending=False)
display(SEGS.round(3))
assert SEGS.customers.sum() == 2172 and round(SEGS.revenue.sum(), 2) == 11009308.50
sim = []
for _ in range(100):
    sh = o.assign(customer_id=rng.permutation(o.customer_id.values))
    sg = rfm(sh).groupby("segment").agg(n=("m", "size"), rev=("m", "sum"))
    sim.append((sg.rev / sg.rev.sum()).reindex(SEGS.index).fillna(0).values)
sim = np.array(sim)
champ_obs = SEGS.loc["Champions", "rev_share"]; champ_sim = sim[:, list(SEGS.index).index("Champions")]
print(f"Champions revenue share observed {champ_obs:.1%} | random assignment mean {champ_sim.mean():.1%} (95% range {np.percentile(champ_sim, 2.5):.1%} to {np.percentile(champ_sim, 97.5):.1%})")

,customers,revenue,orders,recency,cust_share,rev_share
segment,,,,,,
Champions,481,4045957.28,3.736,57.241,0.221,0.368
Loyal,450,2772066.80,2.620,134.400,0.207,0.252
At risk,372,2307486.36,2.597,329.696,0.171,0.210
Hibernating,497,1095024.92,1.139,399.543,0.229,0.099
New / recent,232,496955.30,1.336,63.409,0.107,0.045
Needs attention,140,291817.84,1.300,173.864,0.064,0.027


Champions revenue share observed 36.8% | random assignment mean 36.8% (95% range 35.0% to 38.7%)


In [15]:
fig, (a1, a2) = new_fig(h=4.9, ncols=2, gridspec_kw={"width_ratios": [1.15, 1]}, left=0.13)
y = np.arange(len(SEGS)); h_ = .36
a1.barh(y - h_ / 2, SEGS.cust_share, height=h_, color=GREY, label="share of customers")
a1.barh(y + h_ / 2, SEGS.rev_share, height=h_, color=GREEN, label="share of revenue")
for i, (n, r) in enumerate(SEGS.iterrows()):
    a1.text(r.rev_share + .006, i + h_ / 2, f"{r.rev_share:.1%}", va="center", fontsize=8.5, fontweight="bold")
    a1.text(r.cust_share + .006, i - h_ / 2, f"{r.cust_share:.1%}  ({int(r.customers)})", va="center", fontsize=8, color=MUTED)
    lo, hi = np.percentile(sim[:, list(SEGS.index).index(n)], [2.5, 97.5])
    a1.plot([lo, hi], [i + h_ / 2 + .27] * 2, color=INK, lw=1.2); a1.plot([lo, lo], [i + h_ / 2 + .22, i + h_ / 2 + .32], color=INK, lw=1); a1.plot([hi, hi], [i + h_ / 2 + .22, i + h_ / 2 + .32], color=INK, lw=1)
a1.set_yticks(y, SEGS.index); a1.invert_yaxis(); a1.set_xlim(0, .5); a1.xaxis.set_visible(False); a1.spines[["bottom", "left"]].set_visible(False); a1.tick_params(length=0)
a1.legend(loc="lower right"); a1.set_title("Segment size vs revenue (black line = range under random assignment)", fontsize=9.5)
hm = rf.pivot_table(index="R", columns="F", values="m", aggfunc="mean").sort_index(ascending=False)
cn = rf.pivot_table(index="R", columns="F", values="m", aggfunc="size").sort_index(ascending=False)
sns.heatmap(hm / 1e3, ax=a2, cmap=SEQ_LIGHT, vmin=0, annot=(hm / 1e3).map(lambda v: f"{v:.1f}K") + "\nn=" + cn.astype(str), fmt="", annot_kws={"fontsize": 7.5}, linewidths=.6, linecolor="white", cbar_kws={"shrink": .7, "label": "mean revenue per customer, ₹ thousand"})
a2.set_xlabel("Frequency score (5 = most orders)"); a2.set_ylabel("Recency score (5 = most recent)"); a2.tick_params(length=0); a2.set_title("Revenue per customer by R and F score", fontsize=9.5)
headline(fig, f"Champions are {SEGS.loc['Champions', 'cust_share']:.0%} of customers and {champ_obs:.0%} of revenue; random assignment gives {champ_sim.mean():.0%}, so the segments are partly mechanical",
         "RFM quintile scores for 2,172 ordering customers, grouped into six segments; the black line is the 95% range when orders are randomly reassigned to customers")
footnote(fig, "Source: data/clean fact_orders (revenue = order_total). Recency measured to 4 Nov 2024. Frequency drives the segments; with orders assigned at random a similar split appears, so test RFM targeting on real data before acting.")
print(save(fig, "ds/rfm_segments"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\rfm_segments.png


In [16]:
cr = rf.m.sort_values(ascending=False).values
cumshare = np.cumsum(cr) / cr.sum(); nc = len(cr); xx = np.arange(1, nc + 1) / nc
def gini(v):
    v = np.sort(np.asarray(v, float)); n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1) @ v / (n * v.sum()))
g_obs = gini(cr); g_ci = boot_ci(cr, gini, n=400)
top10, top20 = cumshare[int(nc * .1) - 1], cumshare[int(nc * .2) - 1]
sims, simg, simo = [], [], []
for _ in range(200):
    # every customer keeps at least one order (as in the data), the rest are assigned at random
    perm = rng.permutation(N); cid = np.empty(N, dtype=int); cid[perm[:nc]] = np.arange(nc); cid[perm[nc:]] = rng.integers(0, nc, N - nc)
    rv = np.bincount(cid, weights=o.order_total.values, minlength=nc)
    simg.append(gini(rv)); sims.append(np.cumsum(np.sort(rv)[::-1]) / rv.sum()); simo.append(np.bincount(np.bincount(cid, minlength=nc), minlength=14)[:14] / nc)
sims = np.array(sims); simo = np.array(simo)
opc = rf.f.value_counts().sort_index(); opc = opc.reindex(range(1, 14), fill_value=0) / nc
print(f"top 10% {top10:.1%}, top 20% {top20:.1%}, Gini {g_obs:.3f} (95% CI {g_ci[0]:.3f} to {g_ci[1]:.3f}) | random assignment Gini {np.mean(simg):.3f} | var/mean of orders per customer {rf.f.var():.2f}/{rf.f.mean():.2f}")
assert round(top20, 3) == .409 and round(top10, 3) == .240 and round(g_obs, 3) == .366
fig, (a1, a2) = new_fig(h=4.8, ncols=2)
a1.fill_between(xx, np.percentile(sims, 2.5, axis=0), np.percentile(sims, 97.5, axis=0), color=LTGREY, alpha=.9, label="random assignment (95% range)", lw=0)
a1.plot(xx, cumshare, color=GREEN, lw=2.5, label="observed"); a1.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1)
a1.scatter([.2], [top20], color=GREEN, zorder=5); a1.annotate(f"top 20% of customers = {top20:.1%} of revenue\n(top 10% = {top10:.1%}; real retail is 60 to 80%)", (.2, top20), xytext=(.33, .30), fontsize=9, arrowprops=dict(arrowstyle="-", color=MUTED))
a1.set_xlabel("Share of customers, biggest first"); a1.set_ylabel("Cumulative share of revenue"); a1.set_xlim(0, 1); a1.set_ylim(0, 1.02); clean_axes(a1, "both"); a1.legend(loc="lower right")
a1.set_title(f"Pareto curve: Gini {g_obs:.3f} (95% CI {g_ci[0]:.2f} to {g_ci[1]:.2f}); random {np.mean(simg):.3f}")
xs = np.arange(1, 10)
a2.bar(xs - .2, opc.loc[1:9].values * 100, width=.4, color=GREEN, label="observed")
a2.bar(xs + .2, simo[:, 1:10].mean(0) * 100, width=.4, color=LTGREY, label="random assignment")
a2.set_xticks(xs); a2.set_xlabel("Orders per customer"); a2.set_ylabel("% of ordering customers"); clean_axes(a2); a2.legend()
a2.set_title(f"Fewer heavy buyers than chance: variance {rf.f.var():.2f} vs mean {rf.f.mean():.2f}")
headline(fig, f"Revenue concentration is close to random: the top 20% of customers hold {top20:.0%} of revenue (Gini {g_obs:.2f} vs {np.mean(simg):.2f} random), far below the 60 to 80% of real retail",
         "Customer revenue concentration (left) and the distribution of orders per customer (right) against 200 random reassignments of the same 5,000 orders")
footnote(fig, "Source: data/clean fact_orders, 2,172 ordering customers. Random band: 200 reassignments in which every customer keeps at least one order. The observed curve sits on or just above that band: there are no 'whales'. Orders per customer is under-dispersed (variance below mean), unlike real customer bases.")
print(save(fig, "ds/pareto_customers"))

top 10% 24.0%, top 20% 40.9%, Gini 0.366 (95% CI 0.355 to 0.376) | random assignment Gini 0.347 | var/mean of orders per customer 1.55/2.30


D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\pareto_customers.png


**So what.** Behavioural RFM gives groups that are usable on paper (a third of customers carry about half of revenue), but they are about what a random assignment of the same orders would produce.
Use RFM in CRM (replacing the stored segment, which is null) but treat the segment gaps as hypotheses until real data confirm them.

## D. Basket and price
**Method.** Discount depth = 1 - price / MRP (mean 27.7%, range 15 to 40%) in quintiles; mean quantity, order value and stored margin % per quintile with bootstrap 95% CIs, Spearman correlations;
category mix by stored segment as deviations from the overall mix, with a chi-square test on order counts.

In [17]:
o["disc_q"] = pd.qcut(o.discount, 5, labels=[f"Q{i}" for i in range(1, 6)])
qlab = o.groupby("disc_q", observed=True).discount.agg(["min", "max"]).map(lambda v: f"{v:.0%}")
qlab = [f"Q{i + 1}\n{a}-{b}" for i, (a, b) in enumerate(zip(qlab["min"], qlab["max"]))]
rho_q, p_q = stats.spearmanr(o.discount, o.quantity); rho_t, p_t = stats.spearmanr(o.discount, o.order_total); rho_m, _ = stats.spearmanr(o.discount, o.margin_percentage)
print(f"discount mean {o.discount.mean():.1%} | Spearman with quantity {rho_q:+.3f} (p = {p_q:.2f}), order total {rho_t:+.3f} (p = {p_t:.2f}), margin % {rho_m:+.2f}")
assert abs(rho_q - .006) < .002 and abs(p_q - .66) < .02
fig, axs = new_fig(h=4.5, ncols=3)
spec = [("quantity", "Units per order (mean, 95% CI)", "{:.2f}", rho_q, p_q), ("order_total", "Order value, ₹ (mean, 95% CI)", "{:,.0f}", rho_t, p_t), ("margin_percentage", "Stored margin % of the product (mean, 95% CI)", "{:.1f}", rho_m, 0)]
for ax, (col, ttl, fm, rho, p) in zip(axs, spec):
    g = o.groupby("disc_q", observed=True)[col]
    mu = g.mean(); ci_ = np.array([boot_ci(v.values, n=800) for _, v in g])
    ax.bar(range(5), mu, color=GREEN if col != "margin_percentage" else GREY, width=.62, yerr=[mu - ci_[:, 0], ci_[:, 1] - mu], error_kw=dict(ecolor=INK, capsize=3, elinewidth=1))
    for i, v in enumerate(mu): ax.text(i, ci_[i, 1] * 1.015, fm.format(v), ha="center", fontsize=8.5)
    ax.set_xticks(range(5), qlab, fontsize=8); ax.set_ylim(0, ci_[:, 1].max() * 1.15); ax.yaxis.set_visible(False); ax.spines[["left"]].set_visible(False); ax.tick_params(length=0)
    ax.set_title(ttl + (f"\nSpearman {rho:+.3f}, {pfmt(p)}" if col != "margin_percentage" else f"\nSpearman {rho:+.2f}: margin % is built from discount depth"), fontsize=9)
headline(fig, "Deeper discounts do not sell more or bigger baskets: quantity and order value are flat across discount-depth quintiles",
         "Discount depth quintiles (share off MRP, 15% to 40%). The stored margin % rises with depth only because it is derived from it")
footnote(fig, "Source: data/clean fact_orders, dim_product. Bars start at zero. Order value does not reconcile with item lines (audit), so basket analysis is limited to quantity (1 to 3, uniform) and the order header.")
print(save(fig, "ds/discount_depth"))

discount mean 27.7% | Spearman with quantity +0.006 (p = 0.66), order total -0.015 (p = 0.29), margin % +0.98


D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\discount_depth.png


In [18]:
cs_ = pd.crosstab(o.category, o.segment); chi2, p_cs, dof, _ = stats.chi2_contingency(cs_); V = np.sqrt(chi2 / (cs_.values.sum() * (min(cs_.shape) - 1)))
lv = o.pivot_table(index="category", columns="segment", values="line_value", aggfunc="sum")
seg_share = lv / lv.sum(); overall_mix = o.groupby("category").line_value.sum() / o.line_value.sum()
dev = (seg_share.sub(overall_mix, axis=0) * 100)[["New", "Regular", "Premium", "Inactive"]].loc[overall_mix.sort_values(ascending=False).index]
print(f"category x segment chi-square p = {p_cs:.2f}, Cramer's V = {V:.3f} | largest deviation {dev.abs().max().max():.1f} points")
assert abs(p_cs - .16) < .02
fig, ax = new_fig(h=5.4, left=0.17)
sns.heatmap(dev, ax=ax, cmap=DIV, center=0, vmin=-6, vmax=6, annot=dev.map(lambda v: f"{v:+.1f}"), fmt="", annot_kws={"fontsize": 8.5}, linewidths=.7, linecolor="white", cbar_kws={"shrink": .75, "label": "points vs overall category share"})
ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(length=0); plt.setp(ax.get_yticklabels(), rotation=0); ax.xaxis.tick_top()
ax.set_yticklabels([f"{c_} ({overall_mix[c_]:.1%})" for c_ in dev.index], rotation=0)
headline(fig, f"Every stored segment buys the same category mix: no cell is more than {dev.abs().max().max():.1f} points from the overall share (chi-square {pfmt(p_cs)})",
         "Share of line value by category within each stored segment minus the overall share, in percentage points; overall share in brackets")
footnote(fig, f"Source: data/clean fact_orders, dim_customer. Cramer's V = {V:.2f} (negligible). Segment groups hold 1,000 to 1,500 orders, so a gap of 2 to 3 points is within sampling noise. Line value basis (quantity x unit price).")
print(save(fig, "ds/category_by_segment"))

category x segment chi-square p = 0.16, Cramer's V = 0.050 | largest deviation 1.5 points


D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\category_by_segment.png


**So what.** Neither price depth nor segment shapes what customers buy here. A discount-depth test (cut the discount by 2 points in a few categories) is the way to learn the true response; this data cannot.

## E. Marketing funnel and efficiency
**Method.** Impressions to clicks to conversions by channel; CTR and click-to-conversion rates with Wilson 95% CIs, cost per conversion with a bootstrap CI; Spearman correlations between funnel stages and revenue.

In [19]:
chn = mk.groupby("channel")[["impressions", "clicks", "conversions", "spend", "attributed_revenue"]].sum().loc[["App", "Email", "SMS", "Social Media"]]
tot = chn.sum()
chn["ctr"] = chn.clicks / chn.impressions; chn["cvr"] = chn.conversions / chn.clicks; chn["cpa"] = chn.spend / chn.conversions
ctr_ci = wilson(chn.clicks, chn.impressions); cvr_ci = wilson(chn.conversions, chn.clicks)
def cpa_ci(d, n=1000):
    sp, cv = d.spend.values, d.conversions.values; idx = rng.integers(0, len(d), (n, len(d)))
    return np.percentile(sp[idx].sum(1) / cv[idx].sum(1), [2.5, 97.5])
cpa_c = np.array([cpa_ci(d) for _, d in mk.groupby("channel")])[[0, 1, 2, 3]]
rho_cr, p_cr = stats.spearmanr(mk.conversions, mk.attributed_revenue)
ctr_all, cvr_all, cpa_all = tot.clicks / tot.impressions, tot.conversions / tot.clicks, tot.spend / tot.conversions
print(f"CTR {ctr_all:.1%}, click-to-conversion {cvr_all:.1%}, cost per conversion ₹{cpa_all:.1f}, revenue per conversion ₹{tot.attributed_revenue / tot.conversions:.0f} | conversions vs revenue Spearman {rho_cr:+.3f} (p = {p_cr:.2f}) | conversions per order {tot.conversions / N:.1f}")
assert round(ctr_all, 3) == .101 and round(cvr_all, 3) == .100 and round(cpa_all, 1) == 54.8

CTR 10.1%, click-to-conversion 10.0%, cost per conversion ₹54.8, revenue per conversion ₹108 | conversions vs revenue Spearman -0.011 (p = 0.43) | conversions per order 59.6


In [20]:
fig, axs = new_fig(h=4.6, ncols=4, gridspec_kw={"width_ratios": [1.5, 1, 1, 1]}, left=0.12)
a = axs[0]; vals = [tot.impressions, tot.clicks, tot.conversions]; labs = ["Impressions", "Clicks", "Conversions"]
a.barh(range(3), vals, color=[GREY, MIDGREEN, GREEN], height=.6)
for i, v in enumerate(vals):
    a.text(v + tot.impressions * .015, i, f"{v / 1e6:.2f} M" if v > 1e6 else f"{v / 1e3:,.0f} K", va="center", fontsize=9, fontweight="bold")
a.text(tot.impressions * .52, .5, f"{ctr_all:.1%} click", ha="center", fontsize=9, color=MUTED); a.text(tot.impressions * .52, 1.5, f"{cvr_all:.1%} convert", ha="center", fontsize=9, color=MUTED)
a.set_yticks(range(3), labs); a.invert_yaxis(); a.set_xlim(0, tot.impressions * 1.25); a.xaxis.set_visible(False); a.spines[["bottom", "left"]].set_visible(False); a.tick_params(length=0); a.set_title("All channels, true scale")
for ax, vals_, lo_, hi_, ttl, fm in [(axs[1], chn.ctr, ctr_ci[0], ctr_ci[1], "Click-through rate", "{:.1%}"), (axs[2], chn.cvr, cvr_ci[0], cvr_ci[1], "Click to conversion", "{:.1%}"), (axs[3], chn.cpa, cpa_c[:, 0], cpa_c[:, 1], "Cost per conversion, ₹", "{:.1f}")]:
    v = np.asarray(vals_); lo_, hi_ = np.asarray(lo_), np.asarray(hi_)
    ax.bar(range(4), v, color=GREEN, width=.62, yerr=[v - lo_, hi_ - v], error_kw=dict(ecolor=INK, capsize=3, elinewidth=1))
    for i in range(4): ax.text(i, hi_[i] * 1.02, fm.format(v[i]), ha="center", fontsize=8.5)
    ax.set_xticks(range(4), ["App", "Email", "SMS", "Social"], fontsize=8); ax.set_ylim(0, hi_.max() * 1.18); ax.yaxis.set_visible(False); ax.spines["left"].set_visible(False); ax.tick_params(length=0); ax.set_title(ttl, fontsize=9.5)
headline(fig, f"The funnel is identical in every channel ({ctr_all:.1%} click, {cvr_all:.1%} convert, ₹{cpa_all:.0f} per conversion) and conversions do not predict revenue (rho {rho_cr:+.2f})",
         f"Impressions to clicks to conversions, 5,400 campaign records. Bars with 95% CIs; zero baselines. Conversions vs attributed revenue per record: Spearman {rho_cr:+.3f}, {pfmt(p_cr)}")
footnote(fig, f"Source: data/clean fact_marketing. {tot.conversions / N:.0f} conversions per order in the order data and attributed revenue 2.9x order revenue mean the funnel is not linked to real orders. Report it, but do not optimise on it.")
print(save(fig, "ds/marketing_funnel"))

D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\marketing_funnel.png


## F. Inventory
**Method.** Damage rate = damaged / received by category with Wilson 95% CIs and a chi-square test across categories; received stock compared with units sold in the order data.

In [21]:
iv = inv.merge(pr[["product_id", "category"]], on="product_id")
dm = iv.groupby("category")[["damaged_stock", "stock_received"]].sum(); dm["rate"] = dm.damaged_stock / dm.stock_received
dm["lo"], dm["hi"] = wilson(dm.damaged_stock, dm.stock_received); dm = dm.sort_values("rate", ascending=False)
p_dm = stats.chi2_contingency(np.c_[dm.damaged_stock, dm.stock_received - dm.damaged_stock])[1]
overall_d = iv.damaged_stock.sum() / iv.stock_received.sum()
sold = o.groupby("category").quantity.sum(); recv = iv.groupby("category").stock_received.sum()
sr = pd.DataFrame({"received": recv, "sold": sold}).sort_values("received", ascending=False); sr["ratio"] = sr.received / sr.sold
print(f"damage {overall_d:.2%} (categories {dm.rate.min():.2%} {dm.rate.idxmin()} to {dm.rate.max():.2%} {dm.rate.idxmax()}), chi-square {pfmt(p_dm)} | received {int(sr.received.sum()):,} vs sold {int(sr.sold.sum()):,} = {sr.received.sum() / sr.sold.sum():.1f}x")
assert round(overall_d, 4) == .0385 and abs(p_dm - .24) < .02 and int(sr.received.sum()) == 28779 and int(sr.sold.sum()) == 10034
fig, (a1, a2) = new_fig(h=4.9, ncols=2, gridspec_kw={"width_ratios": [1, 1.1]}, left=0.17)
y = np.arange(len(dm))
a1.errorbar(dm.rate * 100, y, xerr=[(dm.rate - dm.lo) * 100, (dm.hi - dm.rate) * 100], fmt="o", color=GREEN, capsize=2.5, ms=5)
a1.axvline(overall_d * 100, color=GREY, ls="--", lw=1); a1.text(overall_d * 100, -.9, f"overall {overall_d:.2%}", ha="center", fontsize=8.5, color=MUTED)
a1.set_yticks(y, dm.index); a1.set_ylim(len(dm) - .4, -1.3); a1.set_xlim(0, 7); a1.set_xlabel("Damage rate, % of units received (95% CI)"); clean_axes(a1, "x"); a1.set_title(f"Damage by category: {pfmt(p_dm)}", fontsize=9.5)
y2 = np.arange(len(sr)); hh = .38
a2.barh(y2 - hh / 2, sr.received, height=hh, color=GREEN, label="stock received"); a2.barh(y2 + hh / 2, sr.sold, height=hh, color=YELLOW, label="units sold (order data)")
for i, (_, r) in enumerate(sr.iterrows()): a2.text(r.received + 40, i - hh / 2, f"{r.ratio:.1f}x", va="center", fontsize=8.5, fontweight="bold")
a2.set_yticks(y2, sr.index); a2.invert_yaxis(); a2.set_xlim(0, sr.received.max() * 1.15); a2.xaxis.set_visible(False); a2.spines[["bottom", "left"]].set_visible(False); a2.tick_params(length=0); a2.legend(loc="lower right")
a2.set_title(f"Received vs sold units: {sr.received.sum() / sr.sold.sum():.1f}x overall", fontsize=9.5)
headline(fig, f"Damage is {overall_d:.2%} overall and 3.1% to 4.7% by category, all within noise; stock received is {sr.received.sum() / sr.sold.sum():.1f}x the units sold",
         "Left: damaged / received units by category with 95% Wilson intervals. Right: stock received (inventoryNew) vs units sold in the orders file")
footnote(fig, "Source: data/clean fact_inventory (inventoryNew summed to product x month) and fact_orders. Damaged stock is about ₹5.3 L at list price. The 2.9x gap probably reflects that the order file is a sample; no stock levels exist, so lost sales cannot be separated from waste.")
print(save(fig, "ds/inventory_damage_and_stock"))

damage 3.85% (categories 3.10% Pharmacy to 4.65% Household Care), chi-square p = 0.24 | received 28,779 vs sold 10,034 = 2.9x


D:\Wolf\PBI\BlinkitSales-PBI\docs\ds\inventory_damage_and_stock.png


**So what.** Damage differs by category no more than chance allows (binomial noise per product explains all product spread), so a category-level waste programme would target nothing real here.
Reconciling inbound stock to sales needs real stock levels.

## G. Analyses considered and rejected
| Analysis | Why not |
|---|---|
| Market-basket / affinity (association rules) | One item per order: there are no baskets. |
| Store, rider or zone productivity | Store and rider ids are unique on every order (5,000 distinct), so there is nothing to compare. |
| Geospatial / drive-time model of lateness | Only area names (316, about 16 orders each) and a distance column; no coordinates. Distance is unrelated to lateness (A1). |
| Brand analysis | 267 brands for 268 products; every brand is a single product. |
| Customer churn or lifetime-value prediction | No behavioural signal (flat retention, random orders): a model would learn noise. The cross-validated AUCs in A1 (about 0.5 for late and for poor experience) are the same warning. |
| Sentiment or topic mining of feedback text | 25 template sentences, and sentiment is a deterministic function of rating. |
| Price elasticity / promotion lift | Price varies by product, not over time, and discount depth does not move quantity (D); no promotion calendar exists. |
| Anomaly detection on daily revenue | Monthly CV is 5.5% and the series is stationary white noise (ADF p = 0.001, Ljung-Box p = 0.85): there is nothing to flag. |
| Seasonality decomposition (STL) | 19 full months cover under two cycles; seasonal naive is the worst backtest model (7.1% MAPE). |
| Attribution of orders to campaigns | No key between marketing and orders, and attributed revenue is 2.9x order revenue. |

In [22]:
SUMMARY = {
    "late rate": f"{late_rate:.1%} (no driver: smallest factor p = {min(v[0] for v in tests.values()):.2f}, CV AUC {auc_late:.2f})",
    "poor experience": f"{share['Late only']:.1%} late only, {share['Low rating only']:.1%} low rating only, {share['Both']:.1%} both (independence p = {p_ind:.2f})",
    "revenue YoY (Apr-Oct)": f"{yoy:+.1%}: volume {e_y[0] / 1e5:+.2f} L, mix {e_y[1] / 1e5:+.2f} L, AOV {e_y[2] / 1e5:+.2f} L",
    "retention M+1..M+6": f"{np.mean(list(mean_k.values())):.1%} flat",
    "RFM champions": f"{SEGS.loc['Champions', 'cust_share']:.1%} of customers, {champ_obs:.1%} of revenue (random: {champ_sim.mean():.1%})",
    "Gini / top 20%": f"{g_obs:.3f} / {top20:.1%}",
    "discount vs quantity": f"Spearman {rho_q:+.3f} (p = {p_q:.2f})",
    "funnel": f"CTR {ctr_all:.1%}, CVR {cvr_all:.1%}, cost per conversion ₹{cpa_all:.1f}",
    "damage": f"{overall_d:.2%}, category chi-square {pfmt(p_dm)}",
}
display(pd.Series(SUMMARY, name="key result").to_frame())

,key result
late rate,"30.6% (no driver: smallest factor p = 0.20, CV..."
poor experience,"24.2% late only, 15.2% low rating only, 6.4% b..."
revenue YoY (Apr-Oct),"-4.5%: volume -0.85 L, mix -0.02 L, AOV -0.94 L"
retention M+1..M+6,8.9% flat
RFM champions,"22.1% of customers, 36.8% of revenue (random: ..."
Gini / top 20%,0.366 / 40.9%
discount vs quantity,Spearman +0.006 (p = 0.66)
funnel,"CTR 10.1%, CVR 10.0%, cost per conversion ₹54.8"
damage,"3.85%, category chi-square p = 0.24"
